In [1]:
import os, os.path
import numpy as np
import math
import csv
import cv2

In [2]:
image_source = "E:\\Anime\\Other\\nbyn\\Pics\\" # A directory which stores all the png images to be used in grid
name_source = "E:\\Anime\\Other\\Grid\\Names.txt" # A file which contains the corresponding image names of anime on each line (preferably sorted)
complete_name_source = "E:\\Anime\\Other\\Grid\\Complete_names.txt" # Mapping image names to anime names (can be in any order). Format "<image_name> | <anime_name>"
# "|" character is used as seperator as image_name is a file name which can't contain "|"
destination = "E:\\Anime\\Other\\nbyn\\Output\\"

if not os.path.exists(destination):
    os.mkdir(destination)

blank = "Blank"
names_all = []
complete_names = {blank:blank}
spiral_grid = 1 # Follows Row Major Order if 0 else creates a Counterclockwise Spiral originating from "center" of grid

# All image dimensions should be same (width, height, depth)
width  = 1920
height = 1080
depth = 3

# Crop Image (Put (0,0), (blank_height, blank_width)) respectively for no crop
top_left_pixel     = (27,0)
bottom_right_pixel = (height,width)

# Blank Image (to be put if number of images is not a squares)
blank_image = np.zeros([height,width,depth],dtype=np.uint8)
blank_image.fill(255)

# Generates original as well as resized versions to help in compression
resize_factor = 0.333

In [3]:
with open(name_source) as filenames:
    for line in filenames:
        names_all.append(line.strip())

In [4]:
with open(complete_name_source, 'r') as filenames:
    lines = csv.reader(filenames, delimiter='|')
    for line in lines:
        complete_names[line[0].strip()] = line[1].strip()

In [5]:
n = int(np.ceil(math.sqrt(len(names_all))))
names = names_all[:n*n]

In [6]:
def get_names(names, i):
    if i < len(names):
        return names[i]
    else:
        return blank

In [7]:
grid = np.zeros((n,n),dtype=int)
img_status = np.zeros((n,n),dtype=int)

In [8]:
def right(x,y):
    return (x,y+1)

def left(x,y):
    return (x,y-1)
    
def up(x,y):
    return (x-1,y)

def down(x,y):
    return (x+1,y)
    
def updateGrid(grid, x, y, counter):
    if x>=0 and x<n and y>=0 and y<n:
        grid[x][y]=counter
    return counter+1

In [9]:
direction_mapping = {0:right, 1:up, 2:left, 3:down}

In [10]:
counter = 0
if spiral_grid != 0:
    counter = counter + 1
    x = n//2
    y = (n-1)//2
    i = 0
    while (counter < n*n):
        repeat = (i//2)+1
        for z in range(repeat):
            x,y = direction_mapping[i%4](x,y)
            counter = updateGrid(grid, x, y, counter)
        i = i+1
else:
    counter = 0
    for i in range(n):
        for j in range(n):
            grid[i][j] = counter
            counter = counter + 1
print(grid)

[[36 35 34 33 32 31 30]
 [37 16 15 14 13 12 29]
 [38 17  4  3  2 11 28]
 [39 18  5  0  1 10 27]
 [40 19  6  7  8  9 26]
 [41 20 21 22 23 24 25]
 [42 43 44 45 46 47 48]]


In [11]:
def get_grid(i,j):
    if grid[i][j] < len(names):
        return grid[i][j]
    else:
        return -1

In [12]:
def cropped(im):
    return im[top_left_pixel[0]:bottom_right_pixel[0],top_left_pixel[1]:bottom_right_pixel[1]]

In [13]:
def find_image(i,j):
    path = image_source+get_names(names, get_grid(i,j))+".png"
    if (not os.path.isfile(path) or get_grid(i,j)==-1):
        im = blank_image
    else:
        im = cv2.imread(path, cv2.IMREAD_COLOR)
        img_status[i][j] = 1
    return cropped(im)

In [14]:
def concat_vh(list_2d):
      # return final image
    return cv2.vconcat([cv2.hconcat(list_h) 
                        for list_h in list_2d])

In [15]:
im_grid = [[find_image(i,j) for j in range(n)] for i in range(n)]
output = concat_vh(im_grid)

In [16]:
cv2.imwrite(destination+"N by N.png",output)

True

In [17]:
cv2.imwrite(destination+"N by N.jpg",output)

True

In [18]:
im_compressed = cv2.resize(output, None, fx = resize_factor, fy = resize_factor, interpolation= cv2.INTER_LINEAR)
cv2.imwrite(destination+"N by N compressed.png",im_compressed)
cv2.imwrite(destination+"N by N compressed.jpg",im_compressed)

True

In [19]:
str = "From Left to Right, Top to Bottom\n"
sep = ", "
for i in range(n):
    for j in range(n):
        if img_status[i][j]:
            str += complete_names[get_names(names, get_grid(i,j))] + (sep if j < n-1 else "")
        else:
            str += "_" + (sep if j < n-1 else "")
    str += ("\n" if i < n-1 else "")
print(str)
with open(destination+"Caption.txt", 'w') as file:
    file.write(str)

From Left to Right, Top to Bottom
Ping Pong the Animation, Elfen Lied, Death Parade, Bakemonogatari, Horimiya, Vivy: Fluorite Eye's Song, Chainsaw Man
Mononoke, Fate/Zero, The Promised Neverland, Re:Zero - Starting Life in Another World, Made in Abyss, Mob Psycho 100, 86
Mushi-Shi, Your Name, Death Note, Fullmetal Alchemist: Brotherhood, Attack on Titan, Mushoku Tensei: Jobless Reincarnation, Cyberpunk: Edgerunners
My Dress Up Darling, Land of the Lustrous, Violet Evergarden, Hunter x Hunter, Steins;Gate, Kaguya-sama: Love Is War, Erased
A Silent Voice, One Outs, Vinland Saga, Monster, Code Geass, Odd Taxi, Devilman Crybaby
Weathering with You, One Punch Man, Your Lie in April, Terror in Resonance, Spy x Family, Parasyte: The Maxim, Summer Time Rendering
Anohana: The Flower We Saw That Day, Prison School, Domestic Girlfriend, Love and Lies, Jujutsu Kaisen, Uncle From Another World, Bocchi the Rock!
